# Code 11.6.2: KL estimators and an adaptive KL controller

The first part compares two sample-based estimators of the per-token KL divergence with the exact value, for a random next-token distribution over a 50-token vocabulary and a nearby reference. Both are unbiased. On this example the log-ratio estimator is negative on about half the samples and has a standard deviation about six times larger than the second estimator, which is never negative. (When the two distributions are far apart, the advantage of the second estimator shrinks and can reverse.) The second part implements the controller of Ziegler et al. (2019).


In [ ]:
import torch

torch.manual_seed(0)
logits_pi = torch.randn(50)
logits_ref = logits_pi + 0.3 * torch.randn(50)                # a nearby reference, as in RLHF
p, q = logits_pi.softmax(-1), logits_ref.softmax(-1)
exact = (p * (p.log() - q.log())).sum()

y = torch.multinomial(p, 100_000, replacement=True)          # tokens sampled from the policy
log_ratio = p.log()[y] - q.log()[y]                           # log pi(y) / pi_ref(y)
k1 = log_ratio                                                # unbiased, can be negative
k3 = (torch.exp(-log_ratio) - 1) + log_ratio                  # (rho - 1) - log rho, rho = pi_ref / pi
print(f"exact {exact:.4f} | k1 mean {k1.mean():.4f} std {k1.std():.3f} "
      f"| k3 mean {k3.mean():.4f} std {k3.std():.3f} | k1 < 0 on {(k1 < 0).float().mean():.0%} of samples")

class AdaptiveKLController:
    """Log-space proportional controller that steers the observed KL toward a target."""
    def __init__(self, init_beta=0.1, target=6.0, k_beta=0.1):
        self.beta, self.target, self.k_beta = init_beta, target, k_beta

    def update(self, observed_kl):
        error = max(-0.2, min(0.2, (observed_kl - self.target) / self.target))
        self.beta *= 1 + self.k_beta * error
        return self.beta

ctl = AdaptiveKLController(init_beta=0.1, target=6.0)
for kl_t in [2.0, 4.0, 8.0, 12.0, 6.0]:
    print(f"observed KL {kl_t:5.1f} -> beta {ctl.update(kl_t):.4f}")
